# TASK 1 - Read the data
Let’s start with:
- Importing necessary libraries
- Reading the data

In [17]:
# Import neccessary packages
import pandas as pd
import plotly.graph_objects as go
import dash
from dash import dcc
from dash import html
from dash.dependencies import Input, Output
import plotly.express as px

# URL to the airline dataset
URL = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/Data%20Files/airline_data.csv'

# Read the airline data into pandas dataframe
airline_data = pd.read_csv(
    URL,
    encoding='ISO-8859-1',
    dtype={'Div1Airport': str, 'Div1TailNum': str, 
            'Div2Airport': str, 'Div2TailNum': str})

# TASK 2 - Create dash application and get the layout skeleton

Next, we create a skeleton for our dash application. Our dashboard application layout has three components as seen before:
- Title of the application
- Component to enter input year inside a layout division
- 5 Charts conveying the different types of flight delay

Mapping to the respective Dash HTML tags:
- Title added using `html.H1()` tag
- Layout division added using `html.Div()` and input component added using `dcc.Input()` tag inside the layout division.
- 5 charts split into three segments. Each segment has a layout division added using `html.Div()` and chart added using `dcc.Graph()` tag inside the layout division.

In [18]:
# Create a dash application
app = dash.Dash(__name__)

# Build dash app
app.layout = html.Div(
    children=[
        html.H1(),
        html.Div(['Input Year: ', dcc.Input()], style={'font-size': 30}),
        html.Br(),
        html.Br(),
        html.Div([
            html.Div(),
            html.Div()
        ], style={'display': 'flex'}),

        html.Div([
            html.Div(),
            html.Div()
        ], style={'display': 'flex'}),
        html.Div(style={'width': '65%'})
    ]
)

# TASK 3 - Update layout components
## Application title
- Title as `Flight Delay Time Statistics`, align text as `center`, color as `#503D36`, and font size as `30`.

In [19]:
title = html.H1('Flight Delay Time Statistics', 
        style={'TextAlign': 'center', 'color': '#503D36', 'font-size': 30})

## Input component
- Update **dcc.Input** component `id` as `input-year`, default `value` as `2010`, and `type` as `number`. Use `style` parameter and assign height of the input box to be `35px` and font-size to be `30`.

In [20]:
dcc_input =  html.Div(['Input Year: ', dcc.Input(id='input-year', value='2010', type='number')], 
                      style={'height': '35px', 'font-size': 30})

# Output component - Segment 1
Segment 1 is the first `html.Div()`. We have two inner division where first two graphs will be placed.

In [21]:
html.Div([
            html.Div(),         # 1st inner division
            html.Div()          # 2nd inner division
        ], style={'display': 'flex'})

Div(children=[Div(None), Div(None)], style={'display': 'flex'})

## First inner division
- Add `dcc.Graph()` component.
- Update **dcc.Graph** component `id` as `carrier-plot`.

In [22]:
seg1_first_inner = html.Div(dcc.Graph(id='carrier-plot'))

## Second inner division
- Add `dcc.Graph()` component.
- Update **dcc.Graph** component `id` as `weather-plot`.

In [23]:
seg1_second_inner = html.Div(dcc.Graph(id='weather-plot'))

# Output component - Segment 2
Segment 2 is the second `html.Div()`. We have two inner division where the next two graphs will be placed.

In [24]:
html.Div([
            html.Div(),
            html.Div()
        ], style={'display': 'flex'})

Div(children=[Div(None), Div(None)], style={'display': 'flex'})

## First inner division
- Add `dcc.Graph()` component.
- Update **dcc.Graph** component `id` as `nas-plot`.

In [25]:
seg2_first_inner = html.Div(dcc.Graph(id='nas-plot'))

## Second inner division
- Add `dcc.Graph()` component.
- Update **dcc.Graph** component `id` as `security-plot`.

In [26]:
seg2_second_inner = html.Div(dcc.Graph(id='security-plot'))

# Output component - Segment 3
Segment 3 is the last `html.Div()`.
- Add `dcc.Graph()` component to the first inner division.
- Update **dcc.Graph** component `id` as `late-plot`.

In [27]:
seg3 = html.Div(dcc.Graph(id='late-plot'), style={'width': '65%'})

# TASK 4 - Review and add supporting function

In [28]:
""" 
Compute_info function description: This function takes in airline data and selected year as an input and 
performs computation for creating charts and plots.

Arguments:
    airline_data: Input airline data.
    entered_year: Input year for which computation needs to be performed.
    
Returns:
    Computed average dataframes for carrier delay, weather delay, NAS delay, security delay, and 
late aircraft delay.
"""

def compute_info(airline_data, entered_year):
    # Select data
    df = airline_data[airline_data['Year'] == int(entered_year)]

    # Compute the average delay
    avg_car = df.groupby(['Month', 'Reporting_Airline'])['CarrierDelay'].mean().reset_index()
    avg_weather = df.groupby(['Month', 'Reporting_Airline'])['WeatherDelay'].mean().reset_index()
    avg_NAS = df.groupby(['Month', 'Reporting_Airline'])['NASDelay'].mean().reset_index()
    avg_sec = df.groupby(['Month', 'Reporting_Airline'])['SecurityDelay'].mean().reset_index()
    avg_late = df.groupby(['Month', 'Reporting_Airline'])['LateAircraftDelay'].mean().reset_index()

    return avg_car, avg_weather, avg_NAS, avg_sec, avg_late

# TASK 5 - Add the application callback function
The core idea of this application is to get year as user input and update the dashboard in real-time. We will be using `callback` function for the same.

Steps:
- Define the callback decorator
- Define the callback function that uses the input provided to perform the computation
- Create graph and return it as an output
- Run the application

In [29]:
# Callback decorator
@app.callback([
    Output(component_id='carrier-plot', component_property='figure'),
    Output(component_id='weather-plot', component_property='figure'),
    Output(component_id='nas-plot', component_property='figure'),
    Output(component_id='security-plot', component_property='figure'),
    Output(component_id='late-plot', component_property='figure')
    ], 
    Input(component_id='input-year', component_property='value'))

def get_graph(entered_year):

    # Compute the average value for graph data
    avg_car, avg_weather, avg_NAS, avg_sec, avg_late = compute_info(airline_data, entered_year)

    # Line plot for carrier delay
    carrier_fig = px.line(avg_car, x='Month', y='CarrierDelay', color='Reporting_Airline', title='Average carrier delay time (minutes) by airline')
    
    # Line plot for weather delay
    weather_fig = px.line(avg_weather, x='Month', y='WeatherDelay', color='Reporting_Airline', title='Average weather delay time (minutes) by airline')
    
    # Line plot for nas delay
    nas_fig = px.line(avg_NAS, x='Month', y='NASDelay', color='Reporting_Airline', title='Average NAS delay time (minutes) by airline')
    
    # Line plot for security delay
    sec_fig = px.line(avg_sec, x='Month', y='SecurityDelay', color='Reporting_Airline', title='Average security delay time (minutes) by airline')
    
    # Line plot for late aircraft delay
    late_fig = px.line(avg_late, x='Month', y='LateAircraftDelay', color='Reporting_Airline', title='Average late aircraft weather delay time (minutes) by airline')

    return [carrier_fig, weather_fig, nas_fig, sec_fig, late_fig]

# Run the app
if __name__ == '__main__':
    app.run()

# TASK 6 - Run the application

In [32]:
app.layout = html.Div(
    children=[
        title,
        dcc_input,
        html.Br(),
        html.Br(),
        html.Div([
            seg1_first_inner,
            seg1_second_inner
        ], style={'display': 'flex'}),

        html.Div([
            seg2_first_inner,
            seg2_second_inner
        ], style={'display': 'flex'}),
        seg3
    ]
)

# Run the app
if __name__ == '__main__':
    app.run()